In [ ]:
# TITLE : CONVERT IMG TO TEXT

# ============================================================
# PART 1 - OFFLINE / PADDLE SETTINGS

import os
os.environ["PADDLE_PDX_ENABLE_MKLDNN_BYDEFAULT"] = "0"    # crash or freeze turns off run safely

# ============================================================
# PART 2 - IMPORTS

import tkinter as tk
from tkinter import filedialog, messagebox
from tkinter.scrolledtext import ScrolledText

from PIL import Image, ImageTk
from paddleocr import PaddleOCR

# ============================================================
# PART 3 - APPLICATION COLORS

BG = "#121212"
PANEL = "#1C1C1C"
PANEL_2 = "#202020"
TEXT = "#FFFFFF"
SECONDARY = "#A0A0A0"
BORDER = "#303030"
BUTTON = "#292929"
BUTTON_HOVER = "#353535"
ACCENT = "#4F8CFF"

# ============================================================
# PART 4 - LOAD OCR MODEL

print("\n========================================")
print("          TEXTSCAN")
print("========================================\n")
print("Loading PaddleOCR...\nPlease wait...\n")

try:
    ocr = PaddleOCR(lang="en", enable_mkldnn=False)
    print("PaddleOCR loaded successfully.\n")
except Exception as error:
    print("Could not load PaddleOCR.")
    print(error)
    messagebox.showerror("PaddleOCR Error", str(error))
    raise

# ============================================================
# PART 5 - CREATE MAIN WINDOW

root = tk.Tk()
root.title("TextScan")
root.geometry("1050x680")
root.minsize(850, 550)
root.configure(bg=BG)

# ============================================================
# PART 6 - APPLICATION VARIABLES

selected_image = None
preview_image = None

# ============================================================
# PART 7 - SELECT IMAGE & OCR EXECUTION

def select_image():
    global selected_image, preview_image

    # Open file picker
    file_path = filedialog.askopenfilename(
        title="Select Image",
        filetypes=[("Image Files", "*.jpg *.jpeg *.png *.bmp *.webp")]
    )

    if not file_path:
        return

    selected_image = file_path

    # Display image preview
    try:
        image = Image.open(file_path).copy()
        image.thumbnail((480, 430))                               # Resizes to fit
        preview_image = ImageTk.PhotoImage(image)

        image_label.config(image=preview_image, text="")
    except Exception as error:
        messagebox.showerror("Image Error", str(error))
        return

    # Reset GUI text & status
    text_box.delete("1.0", tk.END)                               # Detecting text
    status_label.config(text="Detecting text...")
    root.update()

    # Automatically run OCR
    try:
        print("\n----------------------------------------")
        print(f"Image selected:\n{file_path}")
        print("----------------------------------------")
        print("Running OCR...\n")

        results = ocr.predict(file_path)
        all_text = []

        # Process OCR results
        for result in results:
            try:
                data = result.json
                if callable(data):
                    data = data()

                if not isinstance(data, dict):
                    continue

                result_data = data.get("res", data)
                texts = result_data.get("rec_texts", [])                  # list
                scores = result_data.get("rec_scores", [])                # num or float

                for index, text in enumerate(texts):
                    text = str(text).strip()
                    if not text:
                        continue

                    # Get confidence score
                    confidence = 1.0
                    if index < len(scores):
                        try:
                            confidence = float(scores[index])
                        except ValueError:
                            confidence = 1.0

                    # Keep text with confidence >= 0.5
                    if confidence >= 0.5:
                        all_text.append(text)

            except Exception as error:
                print(f"OCR result parsing error: {error}")

        # Render extracted text
        if all_text:
            final_text = "\n".join(all_text)
            text_box.insert(tk.END, final_text)                           # GUI text box
            status_label.config(text="Text detected")

            print("\n========== EXTRACTED TEXT ==========")
            print(final_text)
            print("====================================\n")
        else:
            text_box.insert(tk.END, "No text detected.")
            status_label.config(text="No text found")
            print("No text detected.")

    except Exception as error:
        status_label.config(text="OCR failed")
        print(f"\n========== OCR ERROR ==========\n{error}\n===============================\n")
        messagebox.showerror("OCR Error", str(error))

# ============================================================
# PART 8 - COPY TEXT

def copy_text():
    text = text_box.get("1.0", tk.END).strip()
    if not text:
        return

    try:
        root.clipboard_clear()
        root.clipboard_append(text)
        root.update()
        status_label.config(text="Copied")
    except Exception as error:
        messagebox.showerror("Copy Error", str(error))

# ============================================================
# PART 9 - CLEAR APPLICATION

def clear_all():
    global selected_image, preview_image

    selected_image = None
    preview_image = None

    image_label.config(image="", text="No image")
    text_box.delete("1.0", tk.END)
    status_label.config(text="Ready")

# ============================================================
# PART 10 - BUTTON CREATOR

def create_button(parent, text, command):
    return tk.Button(
        parent,
        text=text,
        command=command,
        bg=BUTTON,
        fg=TEXT,
        activebackground=BUTTON_HOVER,
        activeforeground=TEXT,
        relief="flat",
        bd=0,
        font=("Arial", 10, "bold"),
        padx=18,
        pady=9,
        cursor="hand2"
    )

# ============================================================
# PART 11 - HEADER

header = tk.Frame(root, bg=BG)
header.pack(fill="x", padx=30, pady=(25, 15))

title = tk.Label(header, text="TextScan", bg=BG, fg=TEXT, font=("Arial", 24, "bold"))
title.pack(anchor="w")

subtitle = tk.Label(header, text="Extract text from images", bg=BG, fg=SECONDARY, font=("Arial", 10))
subtitle.pack(anchor="w", pady=(3, 0))

# ============================================================
# PART 12 - BUTTON BAR

button_frame = tk.Frame(root, bg=BG)
button_frame.pack(fill="x", padx=30, pady=(0, 15))

select_button = create_button(button_frame, "Select Image", select_image)
select_button.pack(side="left")

copy_button = create_button(button_frame, "Copy", copy_text)
copy_button.pack(side="left", padx=8)

clear_button = create_button(button_frame, "Clear", clear_all)
clear_button.pack(side="left")

# ============================================================
# PART 13 - MAIN CONTENT

content = tk.Frame(root, bg=BG)
content.pack(fill="both", expand=True, padx=30, pady=(0, 20))

# ============================================================
# PART 14 - IMAGE PANEL

image_panel = tk.Frame(content, bg=PANEL, highlightbackground=BORDER, highlightthickness=1)
image_panel.pack(side="left", fill="both", expand=True, padx=(0, 8))

image_title = tk.Label(image_panel, text="IMAGE", bg=PANEL, fg=SECONDARY, font=("Arial", 9, "bold"))
image_title.pack(anchor="w", padx=18, pady=(15, 5))

image_label = tk.Label(image_panel, text="No image", bg=PANEL, fg=SECONDARY, font=("Arial", 12))
image_label.pack(fill="both", expand=True, padx=15, pady=15)

# ============================================================
# PART 15 - TEXT PANEL

text_panel = tk.Frame(content, bg=PANEL, highlightbackground=BORDER, highlightthickness=1)
text_panel.pack(side="right", fill="both", expand=True, padx=(8, 0))

text_title = tk.Label(text_panel, text="EXTRACTED TEXT", bg=PANEL, fg=SECONDARY, font=("Arial", 9, "bold"))
text_title.pack(anchor="w", padx=18, pady=(15, 5))

text_box = ScrolledText(
    text_panel,
    bg=PANEL_2,
    fg=TEXT,
    insertbackground=TEXT,
    selectbackground=ACCENT,
    selectforeground=TEXT,
    relief="flat",
    bd=0,
    wrap=tk.WORD,
    font=("Arial", 12)
)
text_box.pack(fill="both", expand=True, padx=15, pady=(5, 15))


# ============================================================
# PART 16 - STATUS BAR

status_label = tk.Label(root, text="Ready", bg=BG, fg=SECONDARY, font=("Arial", 9))
status_label.pack(pady=(0, 12))


# ============================================================
# PART 17 - START APPLICATION

print("Starting TextScan...\n")
root.mainloop()

Creating model: ('PP-LCNet_x1_0_doc_ori', None, None)
Model files already exist. Using cached files. To redownload, please delete the directory manually: `C:\Users\sonag\.paddlex\official_models\PP-LCNet_x1_0_doc_ori`.



          TEXTSCAN

Loading PaddleOCR...
Please wait...



Creating model: ('UVDoc', None, None)
Model files already exist. Using cached files. To redownload, please delete the directory manually: `C:\Users\sonag\.paddlex\official_models\UVDoc`.
Creating model: ('PP-LCNet_x1_0_textline_ori', None, None)
Model files already exist. Using cached files. To redownload, please delete the directory manually: `C:\Users\sonag\.paddlex\official_models\PP-LCNet_x1_0_textline_ori`.
Creating model: ('PP-OCRv6_medium_det', None, None)
Model files already exist. Using cached files. To redownload, please delete the directory manually: `C:\Users\sonag\.paddlex\official_models\PP-OCRv6_medium_det`.
Creating model: ('PP-OCRv6_medium_rec', None, None)
Model files already exist. Using cached files. To redownload, please delete the directory manually: `C:\Users\sonag\.paddlex\official_models\PP-OCRv6_medium_rec`.


PaddleOCR loaded successfully.

Starting TextScan...


----------------------------------------
Image selected:
C:/Users/sonag/Desktop/task/res.jpeg
----------------------------------------
Running OCR...

